# RQ2: Attribution Faithfulness Benchmark (Comprehensiveness, Sufficiency & AOPC)
Full-scale evaluation of decision faithfulness via progressive Top-$k$ WordUnit removal and retention (`deletion` & `embedding_mask` operators) on SST-2 (`Qwen/Qwen2.5-7B` & `Qwen/Qwen2.5-3B`) using empirically calibrated Pareto-optimal hyperparameters:
- **Occlusion (Leave-One-Out)**: deterministic
- **Gradient $\times$ Input**: deterministic
- **Integrated Gradients**: $m = 100$ Riemann steps
- **LIME**: $N = 1000$ neighborhood coalitions
- **SHAP (Permutation)**: $B = 2500$ evaluation budget

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

    GITHUB_TOKEN = "" #@param {type:"string"}
    REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
    BRANCH = "main" #@param {type:"string"}

    try:
        from google.colab import userdata
        token_from_secrets = userdata.get('GH_TOKEN')
        if token_from_secrets:
            GITHUB_TOKEN = token_from_secrets
    except Exception:
        pass

    if GITHUB_TOKEN.strip():
        CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
    else:
        CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

    REPO_DIR = "/content/llm-xai"
    if not os.path.exists(REPO_DIR):
        res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
        print("\n".join(res))
        if not os.path.exists(REPO_DIR):
            raise RuntimeError("Cloning error! Provide Personal Access Token (PAT) in GITHUB_TOKEN or set GH_TOKEN secret.")
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    !pip install -q --upgrade numpy "transformers>=4.43.0,<4.47.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.46.0" scikit-learn scipy pandas matplotlib seaborn huggingface_hub

    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
else:
    DRIVE_RESULTS = "results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    print(f"Running locally. Results directory: {os.path.abspath(DRIVE_RESULTS)}")

### 3. Pipeline Smoke Test (2 samples)

In [ ]:
smoke_model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "microsoft/phi-2"]

!python scripts/run_rq2.py \
    --model {smoke_model} \
    --samples 2 \
    --operator deletion \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --max-evals-shap 100 \
    --lime-samples 100 \
    --ig-steps 10 \
    --output-dir /tmp/smoke_test_rq2 \
    --no-resume

### 4. Main RQ2 Faithfulness Experiment (Milestone Auto-Resume: 300 -> 500 -> 872)
Evaluates `deletion` and/or `embedding_mask` operators across the exact same SST-2 sentences (`seed=42`) as RQ1.

In [ ]:
#@title Main RQ2 Faithfulness Parameters
model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "microsoft/phi-2"]
operator_mode = "both" #@param ["both", "deletion", "embedding_mask"]
milestones = "300" #@param ["300", "300,500", "500", "872"]
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

shap_max_evals = 1200 #@param {type:"integer"}
lime_samples = 500 #@param {type:"integer"}
ig_steps = 50 #@param {type:"integer"}

max_run_samples = 0 #@param {type:"integer"}
auto_shutdown = True #@param {type:"boolean"}

tag = model.split("/")[-1]
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
operators_list = ["deletion", "embedding_mask"] if operator_mode == "both" else [operator_mode]
target_list = [int(x.strip()) for x in milestones.split(",") if x.strip()]

for target_n in target_list:
    for op in operators_list:
        output_dir = f"{DRIVE_RESULTS}/rq2_faithfulness_{tag}_{op}"
        print(f"\n{'='*75}")
        print(f"  RQ2 FAITHFULNESS MILESTONE: N = {target_n} | OP: {op} | MODEL: {model}")
        print(f"  Output Directory: {output_dir}")
        print(f"{'='*75}\n")
        !python scripts/run_rq2.py \
            --model {model} \
            --samples {target_n} \
            --seed {seed} \
            --operator {op} \
            --methods {methods} \
            --max-evals-shap {shap_max_evals} \
            --lime-samples {lime_samples} \
            --ig-steps {ig_steps} \
            {limit_flag} \
            --output-dir {output_dir}

if auto_shutdown and IN_COLAB:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()

### 5. Results Summary Table & Faithfulness Degradation Curves

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

for op in ["deletion", "embedding_mask"]:
    op_dir = f"{DRIVE_RESULTS}/rq2_faithfulness_{tag}_{op}"
    summary_csv = os.path.join(op_dir, "rq2_faithfulness_summary.csv")
    if os.path.exists(summary_csv):
        print(f"\n=== RQ2 Faithfulness Summary ({tag} | Operator: {op}) ===")
        df_sum = pd.read_csv(summary_csv)
        display(df_sum.round(4))
        plot_path = os.path.join(op_dir, "plots", "faithfulness_curves.png")
        if os.path.exists(plot_path):
            display(Image(plot_path, width=750))